In [81]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import math
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB

In [65]:
data = pd.read_csv("agaricus-lepiota.data", header = None)

In [66]:
X = data.iloc[:, 1:]
y = data.iloc[:, 0]

## Problem 4

In [67]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    test_size = 0.25, 
    random_state = 42, 
    stratify = y
)

### Part 1 

In [68]:
prior = y_train.value_counts(normalize=True).to_dict()

classes = y_train.unique()

conditional_probs = {}

for col in X_train.columns:
    conditional_probs[col] = {}
    
    values = X_train[col].unique()
    k = len(values)
    
    for c in classes:
        conditional_probs[col][c] = {}
        
        subset = X_train.loc[y_train == c]
        counts = subset[col].value_counts()
        total = len(subset)
        
        for v in values:
            count_v = counts.get(v, 0)
            prob = (count_v + 1) / (total + k)
            conditional_probs[col][c][v] = prob

In [69]:
conditional_probs[1]['e']

{'x': np.float64(0.4683744465528147),
 'k': np.float64(0.05471220746363061),
 'f': np.float64(0.37507906388361795),
 'b': np.float64(0.09361163820366857),
 's': np.float64(0.007906388361796331),
 'c': 0.00031625553447185326}

### Part 2

In [70]:
def predict_proba(row, prior, conditional_probs, classes):
    log_probs = {}
    
    for c in classes:
        log_prob = math.log(prior[c])
        
        for col in row.index:
            value = row[col]
            log_prob += math.log(conditional_probs[col][c][value])
        
        log_probs[c] = log_prob
    
    max_log = max(log_probs.values())
    
    exp_probs = {}
    total = 0
    
    for c in classes:
        exp_probs[c] = math.exp(log_probs[c] - max_log)
        total += exp_probs[c]
    
    for c in classes:
        exp_probs[c] /= total
    
    return exp_probs

In [71]:
test_probs = []

for _, row in X_test.iterrows():
    probs = predict_proba(row, prior, conditional_probs, classes)
    test_probs.append(probs)

In [72]:
test_probs[0]

{'p': 3.6006290577338105e-08, 'e': 0.9999999639937094}

### Part 3

In [77]:
y_pred = []

for probs in test_probs:
    predicted_class = max(probs, key=probs.get)
    y_pred.append(predicted_class)

In [78]:
# Accuracy (overall)
accuracy = accuracy_score(y_test, y_pred)

# Poisonous
precision_p = precision_score(y_test, y_pred, pos_label='p')
recall_p = recall_score(y_test, y_pred, pos_label='p')
f1_p = f1_score(y_test, y_pred, pos_label='p')

# Edible
precision_e = precision_score(y_test, y_pred, pos_label='e')
recall_e = recall_score(y_test, y_pred, pos_label='e')
f1_e = f1_score(y_test, y_pred, pos_label='e')

print("Accuracy:", accuracy)

print("\nPoisonous:")
print("Precision:", precision_p)
print("Recall:", recall_p)
print("F1:", f1_p)

print("\nEdible:")
print("Precision:", precision_e)
print("Recall:", recall_e)
print("F1:", f1_e)

Accuracy: 0.9527326440177253

Poisonous:
Precision: 0.9911012235817576
Recall: 0.9101123595505618
F1: 0.9488817891373802

Edible:
Precision: 0.9222614840989399
Recall: 0.9923954372623575
F1: 0.9560439560439561


### Part 4

In [87]:
X_train_enc = X_train.copy()
X_test_enc = X_test.copy()

encoders = {}

for col in X_train.columns:
    le = LabelEncoder()
    X_train_enc[col] = le.fit_transform(X_train[col])
    X_test_enc[col] = le.transform(X_test[col])
    encoders[col] = le

In [88]:
nb_model = MultinomialNB()

nb_model.fit(X_train_enc, y_train)
y_pred_nb = nb_model.predict(X_test_enc)

In [89]:
# Accuracy
accuracy_nb = accuracy_score(y_test, y_pred_nb)

# Poisonous ('p')
precision_p_nb = precision_score(y_test, y_pred_nb, pos_label='p')
recall_p_nb = recall_score(y_test, y_pred_nb, pos_label='p')
f1_p_nb = f1_score(y_test, y_pred_nb, pos_label='p')

# Edible ('e')
precision_e_nb = precision_score(y_test, y_pred_nb, pos_label='e')
recall_e_nb = recall_score(y_test, y_pred_nb, pos_label='e')
f1_e_nb = f1_score(y_test, y_pred_nb, pos_label='e')

print("MultinomialNB Results:")

print("Accuracy:", accuracy_nb)

print("\nPoisonous:")
print("Precision:", precision_p_nb)
print("Recall:", recall_p_nb)
print("F1:", f1_p_nb)

print("\nEdible:")
print("Precision:", precision_e_nb)
print("Recall:", recall_e_nb)
print("F1:", f1_e_nb)

MultinomialNB Results:
Accuracy: 0.8138847858197932

Poisonous:
Precision: 0.899070385126162
Recall: 0.6915219611848825
F1: 0.7817551963048499

Edible:
Precision: 0.7636932707355243
Recall: 0.9277566539923955
F1: 0.8377682403433476
